# Solutions to Exercises 12: More SPARQL

1. Load in the ontology `teaching.rdf` and 

a) Write a query using the OWLReady2 search capability to find all entities to which the entity with attribute `module_title="ECS8052 Knowledge Engineering` has a *direct* relation (in either direction). You will need to use the ontology to help you with this. 


A `Module` can be involved in the following relations:

* `Requires(Course, Module)`
* `Teaches(Staff,Module)`
* `Passed(Student,Module)`

The other relation present in the ontology is 

* `Enrolled_(Student,Course)`


In [1]:
from owlready2 import *
onto = get_ontology("teaching.owl").load()
module = onto.search_one(module_title="Machine Learning")
students = onto.search(Passed = module)
courses = onto.search(Requires = module)
staff = onto.search(Teaches = module)
print(students)
print(courses)
print(staff)


[teaching.stu01, teaching.stu02, teaching.stu03, teaching.stu04, teaching.stu05, teaching.stu06, teaching.stu07, teaching.stu08, teaching.stu09, teaching.stu10, teaching.stu11, teaching.stu12, teaching.stu13, teaching.stu14, teaching.stu15]
[teaching.pro01, teaching.pro02, teaching.pro03]
[teaching.sta04]



    b) Repeat part (a) using SPARQL.

A key thing to notice here is that `Module` has only incoming relationships

In [ ]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT DISTINCT ?x ?r ?name WHERE
    {
        ?m rdf:type ONTO:Module
        ?m ONTO:module_title ?t
        FILTER(STRENDS(?t, "Knowledge Engineering"))
        {
            ?m ?r ?x
            FILTER(STRSTARTS(STR(?r), "http://www.dummy.info/new.owl#"))
            FILTER(STRSTARTS(STR(?x), "http://www.dummy.info/new.owl#"))
            BIND("Links from Knowledge Engineering" as ?name)
        }
        UNION
        {
            ?x ?r ?m
            BIND("Links to Knowledge Engineering" as ?name)
        }
    }
    """))

for i in x:
    print(i)
    print(i)

2. Write a SPARQL query to get all students who study a particular module

In [2]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?s
    WHERE {
        ?m rdf:type ONTO:Module
        ?m ONTO:module_title "Knowledge Engineering"
        ?c ONTO:Requires ?m
        ?s ONTO:Enrolled ?c
    } 
    
    """))

for i in x:  
    print(i)

[teaching.stu01]
[teaching.stu02]
[teaching.stu03]
[teaching.stu04]
[teaching.stu05]


3. Write a SPARQL query to get all students who *have not passed* all of their required modules.

*Hint*: get all student-module pairs and then used `FILTER NOT EXISTS` to identify those for which the `Passed` relation does not exist.

In [3]:
x = list(default_world.sparql( 
   """
   PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
   SELECT ?name ?program ?module
   WHERE {
      ?s a ONTO:Student
      ?s ONTO:Enrolled ?c
      ?c ONTO:Requires ?m
      ?s ONTO:person_name ?name
      ?c ONTO:program_title ?program
      ?m ONTO:module_title ?module

      FILTER NOT EXISTS { ?s ONTO:Passed ?m }
   }
   """
   )
)
for i in x:
    print(i)

['Demis Hassabis', 'MSc Artificial Intelligence', 'Knowledge Engineering']
['Martin Hellman', 'MSc Cyber Security', 'Malware Detection']
['John von Neumann', 'MSc Data Analytics and AI', 'Statistics']


4. Write a SPARQL query to return how many students each Professor teaches. Sort them result in descending order by number of students.

*Hint*: use `GROUP BY` and `ORDER BY`

In [4]:
x = list(default_world.sparql( 
   """
   PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
   SELECT ?professor (COUNT(DISTINCT ?s) AS ?students_reached)
    WHERE {
        ?p a ONTO:Professor
        ?p ONTO:person_name ?professor
        ?p ONTO:Teaches ?m
        ?c ONTO:Requires ?m
        ?s ONTO:Enrolled ?c
    }
    GROUP BY ?professor
    ORDER BY DESC(?students_reached) ?professor
   """
   )
)
for i in x:
    print(i)

['Prof Hui Wang', 15]
['Dr Barry Devereux', 10]
['Dr Ihsen Alouani', 10]
['Dr Yang Hua', 10]
['Dr Amy Liu', 5]
['Dr Ayesha Khalid', 5]
['Dr Chao Tian', 5]
['Dr Ciara Rafferty', 5]
['Dr Hannah Mitchell', 5]
['Dr Lisa McFetridge', 5]
['Dr Neil Anderson', 5]
['Dr Niall McLaughlin', 5]
['Dr Reza Rafiee', 5]
['Prof Iain Styles', 5]
["Prof Maire O'Neill", 5]


5. Write a SPARQL query to list very student and professor with a count of how many modules they have passed/teach, so students are counted by modules passed and professors by modules taught. Group the results by role (Professor/Student) and show the Professors first.

In [ ]:
x = list(default_world.sparql( 
   """
   PREFIX ONTO: <http://www.dummy.info/new.owl#>

    SELECT ?name ?role (COUNT(?m) AS ?modules)
    

    WHERE {
        {
            ?p rdf:type ONTO:Student
            ?role rdf:type ONTO:Student
            ?p ONTO:person_name ?name
            ?p ONTO:Passed ?m 
        }
        UNION
        {
            ?p rdf:type ONTO:Professor
            ?role rdf:type ONTO:Professor
            ?p ONTO:person_name ?name
            ?p ONTO:Teaches ?m 
        }
    }
    GROUP BY ?name ?role
    ORDER BY ?role ?name

   """
   )
)
for i in x:
    print(i)

Here is a more elegant solution for this that uses a `FILTER` to ensure that we only select students and Professors, and an `OPTIONAL` block to catch people who do not teach or have not passes any module.

In [ ]:
x = list(default_world.sparql( 
   """
   PREFIX ONTO: <http://www.dummy.info/new.owl#>

    SELECT ?name ?role (COUNT(?m) AS ?modules)
    WHERE {
    ?p a ?role 
    ?p ONTO:person_name ?name 
    FILTER (?role = ONTO:Student || ?role = ONTO:Professor)
    OPTIONAL {
        { ?p ONTO:Passed ?m }
        UNION
        { ?p ONTO:Teaches ?m }
    }
    }
    GROUP BY ?name ?role
    ORDER BY ?role ?name
   """
   )
)
for i in x:
    print(i)